In [2]:
# CS 340 Project Two Dashboard
# Bruno Manuel

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure dashboard component imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd

# Import CRUD Python module
from animal_shelter import AnimalShelter

# Configure JupyterDash
JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to MongoDB through CRUD module
db = AnimalShelter(username, password)

# Load all records at startup
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId column so Dash table does not crash
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

# Debug check
print("Data loaded successfully.")
print("Number of records:", len(df))
print("Columns:", df.columns.tolist())


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())

app.layout = html.Div([

    html.Center([
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image.decode()),
            style={"height": "180px"}
        ),
        html.H1("CS 340 Project Two Dashboard"),
        html.H3("Bruno Manuel - Grazioso Salvare Animal Rescue Dashboard")
    ]),

    html.Hr(),

    html.Div([
        html.H4("Select Rescue Type"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "wilderness"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"}
            ],
            value="reset",
            labelStyle={
                "display": "inline-block",
                "margin-right": "25px",
                "font-size": "16px"
            }
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            } for i in df.columns
        ],

        data=df.to_dict("records"),

        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "220px",
            "whiteSpace": "normal",
            "fontFamily": "Arial",
            "fontSize": "12px"
        },

        style_header={
            "fontWeight": "bold",
            "backgroundColor": "#f2f2f2"
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "50%", "padding": "10px"}
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "50%", "padding": "10px"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Update dashboard table based on selected rescue filter."""

    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "wilderness":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))

    if "_id" in filtered_df.columns:
        filtered_df.drop(columns=["_id"], inplace=True)

    return filtered_df.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    """Display a pie chart showing breed distribution."""

    if viewData is None or len(viewData) == 0:
        return [html.H4("No data available for chart.")]

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns:
        return [html.H4("Breed column is not available.")]

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names="breed",
                title="Preferred Animal Breeds"
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected table column."""

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    """Update geolocation map based on selected table row."""

    if viewData is None or len(viewData) == 0:
        return [html.H4("No data available for map.")]

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    # Required map fields
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return [html.H4("Location data is not available.")]

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    breed = dff.loc[row, "breed"] if "breed" in dff.columns else "Unknown Breed"
    name = dff.loc[row, "name"] if "name" in dff.columns else "Unknown Name"

    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H3("Animal Information"),
                            html.P("Name: {}".format(name)),
                            html.P("Breed: {}".format(breed))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server(mode="inline")

Data loaded successfully.
Number of records: 10000
Columns: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']
